<a href="https://colab.research.google.com/github/juli0AND/Evaluacion-comparativa-de-YOLOv5-y-YOLOv8/blob/main/YOLOV8s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# INSTALAR LIBRERÍAS
# ==========================================================
!pip install ultralytics roboflow thop pandas -q

# ==========================================================
# IMPORTAR LIBRERÍAS
# ==========================================================
from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import Image, display
from google.colab import files
from thop import profile

import torch
import time
import numpy as np
import pandas as pd
import os

# ==========================================================
# DESCARGAR DATASET
# ==========================================================
rf = Roboflow(api_key="io4o0d2qhkzDHpqNE3Tb")

project = rf.workspace("deteccion-residuos") \
            .project("new-classification-waste-yolo_small-vaimd")

version = project.version(3)

dataset = version.download("yolov8")
# ==========================================================
# DESACTIVAR WANDB
# ==========================================================
os.environ["WANDB_MODE"] = "disabled"

# ==========================================================
# DISPOSITIVO
# ==========================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

# ==========================================================
# MODELO BASE
# ==========================================================
MODEL_BASE = "yolov8s.pt"
MODEL_NAME = "YOLOv8s_3"

model = YOLO(MODEL_BASE)

# ==========================================================
# ENTRENAMIENTO
# ==========================================================
results = model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=50,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    seed=3,
    name=f"{MODEL_NAME}_train"
)

# ==========================================================
# RUTA REAL DEL EXPERIMENTO
# ==========================================================
exp_path = str(results.save_dir)

print("Experimento:", exp_path)

# ==========================================================
# RUTA DEL BEST.PT
# ==========================================================
WEIGHTS = os.path.join(
    exp_path,
    "weights",
    "best.pt"
)

# ==========================================================
# CARGAR BEST MODEL
# ==========================================================
model = YOLO(WEIGHTS)
model.to(device)

# ==========================================================
# VALIDACIÓN DEL BEST.PT
# ==========================================================
metrics = model.val()

# ==========================================================
# MÉTRICAS
# ==========================================================
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

print("\n======== MÉTRICAS ========")

print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"mAP50     : {map50:.6f}")
print(f"mAP50-95  : {map5095:.6f}")

# ==========================================================
# PARÁMETROS
# ==========================================================
params = sum(
    p.numel()
    for p in model.model.parameters()
)

params_m = params / 1e6

# ==========================================================
# FLOPs
# ==========================================================
dummy = torch.randn(
    1,
    3,
    640,
    640
).to(device)

flops, _ = profile(
    model.model,
    inputs=(dummy,),
    verbose=False
)

flops_g = flops / 1e9

# ==========================================================
# TAMAÑO DEL MODELO
# ==========================================================
size_mb = os.path.getsize(
    WEIGHTS
) / (1024 * 1024)

# ==========================================================
# FPS (FORWARD PASS REAL)
# ==========================================================
model.model.eval()

for _ in range(20):

    with torch.no_grad():
        _ = model.model(dummy)

times = []

for _ in range(100):

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        _ = model.model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    times.append(
        (end - start) * 1000
    )

avg_time = np.mean(times)
std_time = np.std(times)

fps = 1000 / avg_time

# ==========================================================
# RESULTADOS
# ==========================================================
df = pd.DataFrame({

    "Modelo":[MODEL_NAME],

    "Precision":[precision],
    "Recall":[recall],
    "mAP50":[map50],
    "mAP50_95":[map5095],

    "Tiempo_ms":[avg_time],
    "Tiempo_std":[std_time],

    "FPS":[fps],

    "Parametros_M":[params_m],
    "FLOPs_GFLOPs":[flops_g],
    "Tamano_MB":[size_mb]
})

print("\n======== TABLA FINAL ========")
print(df)

# ==========================================================
# EXPORTAR CSV
# ==========================================================
csv_name = f"{MODEL_NAME}_resumen.csv"

df.to_csv(
    csv_name,
    index=False
)

# ==========================================================
# IMÁGENES DEL ENTRENAMIENTO
# ==========================================================
results_path = os.path.join(
    exp_path,
    "results.png"
)

confusion_path = os.path.join(
    exp_path,
    "confusion_matrix.png"
)

# ==========================================================
# MOSTRAR
# ==========================================================
if os.path.exists(results_path):
    display(Image(results_path))

if os.path.exists(confusion_path):
    display(Image(confusion_path))

# ==========================================================
# DESCARGAS
# ==========================================================
files.download(csv_name)
files.download(WEIGHTS)

if os.path.exists(results_path):
    files.download(results_path)

if os.path.exists(confusion_path):
    files.download(confusion_path)